# 23 · PyTorch en pratique 1 : du dataset au modèle sauvegardé — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch23_pytorch_1/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 23.Q1 à 23.Q12, 23.R1 à 23.R3, 23.1 à 23.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 23.12 à 23.16 | l'environnement et les tenseurs, Fashion-MNIST regardé, oublier la conversion ou la normalisation, tout le prétraitement au même endroit, le MLP du livre (*prochaine session de génération*) | | | |
| B | 23.17 à 23.20 | trois bugs d'inférence, ta fonction fit() et son historique, lire l'historique, l'analyse des erreurs (*prochaine session de génération*) | | | |
| C | 23.21 à 23.24 | des images « du monde réel », prédire, reprendre un entraînement interrompu, un modèle pré-entraîné de torchvision (*prochaine session de génération*) | | | |
| D | 23.25 à 23.29 | les callbacks en PyTorch, les plannings de learning rate, la patience de l'early stopping, la fiche modèle, le défi reproductibilité (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Préparer un dataset d'images pour PyTorch : formes `(N, C, H, W)`, type `float32`, normalisation avec les statistiques du train, `Dataset` et `DataLoader`.
- Écrire ta propre fonction `fit()` (boucle, validation, historique), puis analyser les erreurs du modèle et prédire sur de nouvelles images.
- Sauvegarder et recharger un modèle, reprendre un entraînement, réutiliser un modèle pré-entraîné, et écrire en PyTorch les callbacks de Keras (meilleur checkpoint, planning de learning rate, early stopping).

**Rappel express.** Indice plat d'un élément $(i, j, k)$ de forme $(D_1, D_2, D_3)$ : $i\,D_2 D_3 + j\,D_3 + k$. Normalisation : $x' = (x/255 - \mu_{\text{train}})/\sigma_{\text{train}}$. Couche dense : $(n_{in} + 1)\,n_{out}$ paramètres. Cross-entropy sur des logits : $L = -z_y + \ln \sum_j e^{z_j}$. Mini-batches par epoch : $\lceil N/B \rceil$. `LambdaLR` : $\eta_e = \eta_0 \cdot \lambda(e)$. Coût d'une epoch : $\approx 6PN$ opérations.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 23.1 à 23.7 et 🧮 23.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs lettres ou un ordre : entre guillemets (`"EF"`) ; une forme ou un triplet d'indices : entre guillemets (`"(5, 7, 1)"`) ; une liste : entre crochets (`[9, 9, 9]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la réflexion (🗣️ 23.9, ⚖️ 23.10, 📄 23.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np

# Recall 23.R3: five pixels of a tiny training set, already divided by 255
PIXELS_R3 = np.array([0.0, 0.2, 0.4, 0.4, 1.0])
MU_R3, SIGMA_R3 = PIXELS_R3.mean(), PIXELS_R3.std()          # ddof=0, like np.std
SIGMA1_R3 = PIXELS_R3.std(ddof=1)

# Ex 23.1: the letters M to X at positions 0 to 11, reshaped in C order
LETTERS_231 = np.array(list("MNOPQRSTUVWX"))


def flat_index(index, shape):
    """Position in memory (C order) of the element `index` of a tensor of shape `shape`."""
    return int(np.ravel_multi_index(index, shape))


# Ex 23.2: an RGB image of 2 rows and 3 columns, channels_last, x[h, w, c] at position 9h + 3w + c
X_232 = np.arange(18).reshape(2, 3, 3)
Y_232 = X_232.transpose(2, 0, 1)                                # x.permute(2, 0, 1): y[c, h, w] = x[h, w, c]
Z_232 = np.ascontiguousarray(Y_232)                             # y.contiguous()


def dense(n_in, n_out):
    """Parameters of nn.Linear(n_in, n_out): weights and biases."""
    return (n_in + 1) * n_out


def softmax(z):
    e = np.exp(np.asarray(z, float) - max(z))
    return e / e.sum()


def cross_entropy(z, label):
    """What nn.CrossEntropyLoss() returns for one example: -z_y + ln sum_j exp(z_j)."""
    z = np.asarray(z, float)
    return float(-z[label] + np.log(np.exp(z).sum()))


# Ex 23.4: logits of a 3-class classifier, label 1
Z_234 = [1.0, 2.0, 0.0]
P_234 = softmax(Z_234)

# Ex 23.5: 10 % of the 60 000 training images kept for validation, mini-batches of 256
N_235 = 60_000 - 6_000

# Ex 23.6: validation losses of epochs 1 to 16
VAL_236 = [0.60, 0.45, 0.40, 0.41, 0.38, 0.385, 0.379, 0.39, 0.40, 0.41, 0.37, 0.42, 0.43, 0.44, 0.45, 0.46]


def early_stopping(losses, patience, min_delta=0.0):
    """(stop epoch, best epoch), epochs counted from 1; the rule of the statement and of mylearn's EarlyStopping."""
    best, best_epoch, counter = None, None, 0
    for epoch, loss in enumerate(losses, start=1):
        if best is None or loss < best - min_delta:
            best, best_epoch, counter = loss, epoch, 0
        else:
            counter += 1
            if counter >= patience:
                return epoch, best_epoch
    return None, best_epoch


STOP_236A, BEST_236A = early_stopping(VAL_236, 3)
STOP_236C, BEST_236C = early_stopping(VAL_236, 3, 0.005)
STOP_236E, BEST_236E = early_stopping(VAL_236, 5)


# Ex 23.7: the book's schedule, a factor for LambdaLR (epochs counted from 0)
def book_lambda(e):
    return max(0.1, 1 - 0.01 * e)


# Ex 23.8: the book's MLP on 60 000 images, 6PN operations per epoch, a CPU at 50 billion operations per second
P_238, N_238 = dense(784, 784) + dense(784, 10), 60_000
OPS_238 = 6 * P_238 * N_238
SECONDS_238 = OPS_238 / 50e9

**23.Q1 — Pourquoi utiliser une bibliothèque plutôt que tout coder soi-même ?**

In [3]:
wb.record("23.Q1a", False, mistakes={"écrire mylearn t'a appris ce que calculent les fonctions que tu appelles maintenant": True})
wb.record("23.Q1b", "B", mistakes={"les formules sont les mêmes : c'est la façon de les calculer qui change": "A",
          "aucune bibliothèque ne garantit une accuracy : elle dépend des données et du modèle": "C",
          "sans données, il n'y a rien à apprendre, quelle que soit la bibliothèque": "D"})
wb.record("23.Q1c", "C", mistakes={"c'est l'inverse : la bibliothèque fait monter d'un niveau d'abstraction": "A",
          "l'analogie parle de niveau d'abstraction, pas d'une langue à apprendre": "B",
          "l'analogie oppose deux façons d'écrire un programme, pas un plan et une maison": "D"})
wb.record("23.Q1d", True, mistakes={"relis comment les tests de mylearn décident si ta fonction est juste": False})
wb.record("23.Q1e", False, mistakes={"qui choisit les données, le prétraitement, la loss et la façon d'évaluer ?": True})

WB_ANSWER {"hash": "461fc674c4ff4b3994c35eb9605e4e79fb4b9990497725ac571103906a49db60", "id": "23.Q1a", "kind": "bool", "mistakes": {"0797e89ece23d8acfa0775e754ddbfcde94271534cb4d3cc8365a5fc504e3367": "écrire mylearn t'a appris ce que calculent les fonctions que tu appelles maintenant"}}
📝 Ex 23.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "f7a0385db48d26e8e9e24c6728248b50e08d5f6c2df5484c45c6f56b5705c9a4", "id": "23.Q1b", "kind": "str", "mistakes": {"08784da4a62bc769ef51465f7dafed07eaeb11b270ee898297e219c9fd200d47": "aucune bibliothèque ne garantit une accuracy : elle dépend des données et du modèle", "ae9d6dc91922dda29224e963a162d459af9d9f4ccab07db6fff659a18a2076d6": "sans données, il n'y a rien à apprendre, quelle que soit la bibliothèque", "cd7ea7b8fa41c15ef7360820a251f798e99e129bd6e9d498467bec863af5a99e": "les formules sont les mêmes : c'est la façon de les calculer qui change"}}
📝 Ex 23.Q1b : réponse enregistrée (str).
WB_ANSWER {"hash": "34520809cdd75c6da3e98685227e5768478

**23.Q2 — Avertissement ou erreur ? Versions et dépréciations**

In [4]:
wb.record("23.Q2a", "B", mistakes={"un avertissement n'arrête pas le programme": "A",
          "un avertissement ne dit pas que le résultat est faux": "C",
          "une bibliothèque absente lève une ImportError, pas un avertissement": "D"})
wb.record("23.Q2b", True, mistakes={"que fait Python d'une exception que personne ne rattrape ?": False})
wb.record("23.Q2c", False, mistakes={"relis les encadrés 🕰️ : des méthodes et des arguments du livre ont disparu": True})
wb.record("23.Q2d", "D", mistakes={"la dernière version est justement ce qui change d'une année à l'autre": "A",
          "faire taire les avertissements ne fige rien": "B",
          "sans bibliothèque, ton code est plus lent, pas plus reproductible": "C"})
wb.record("23.Q2e", False, mistakes={"tu ne verrais plus aucune annonce de changement à venir": True})

WB_ANSWER {"hash": "ca44e67931542d5e727d66ea041f98a10e2d92486d18bb04537410aff8b3a359", "id": "23.Q2a", "kind": "str", "mistakes": {"10342c830761749b549b9fc9c9de374dd0ab75a654b71e73ed52956e63f6acf0": "un avertissement ne dit pas que le résultat est faux", "137c8fbe17d17b91be5904b73bfdc05d9e9cc733d74620ba08caaed60851799b": "une bibliothèque absente lève une ImportError, pas un avertissement", "fa4d734fd04176251d8c6f9bc7e75482a5b4224b26a6166b1176f3e671e1ba97": "un avertissement n'arrête pas le programme"}}
📝 Ex 23.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "601a38d03e97480977dd49785763f13ad356de363accc1a2d3f007526ce92b38", "id": "23.Q2b", "kind": "bool", "mistakes": {"968941c69e1ffb2b3fc9c46c1738d6b46c8d53ea39d7a14399be6b32acd4e2fd": "que fait Python d'une exception que personne ne rattrape ?"}}
📝 Ex 23.Q2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "9ec5aa995ad1cdb6f23b0d3b5f5eb7927e4acb88932c4fdc4c34aafb65826230", "id": "23.Q2c", "kind": "bool", "mistakes": {"cdf39689f10e

**23.Q3 — Déboguer méthodiquement : réduire, isoler, reconstruire**

In [5]:
wb.record("23.Q3a", "C", mistakes={"le code d'une grande bibliothèque se compte en centaines de milliers de lignes": "A",
          "une erreur de forme ou de type revient à chaque exécution": "B",
          "le même appel fautif planterait aussi avec une autre bibliothèque": "D"})
wb.record("23.Q3b", True, mistakes={"relis la fin du §23.2.2 de la fiche": False})
wb.record("23.Q3c", 32, mistakes={"le premier nombre de mat1 est la taille du mini-batch ; le second, le nombre de features": 784,
          "748 est la première dimension de mat2, la matrice de poids": 748,
          "10 est le nombre de sorties de la dernière couche": 10})
wb.record("23.Q3d", 784, mistakes={"748 est justement la faute de frappe": 748,
          "in_features est le nombre de valeurs reçues par image, pas la taille du mini-batch": 32,
          "10 est le nombre de sorties (out_features)": 10})
wb.record("23.Q3e", False, mistakes={"quel code a fourni les formes qui ne vont pas ensemble ?": True})

WB_ANSWER {"hash": "c6c3301eb25f5020a9c8dbccb7acfd97e125c32cfca8319008cd65e3c915bb9d", "id": "23.Q3a", "kind": "str", "mistakes": {"889c2427024931ebf62ae36cc96f6bf1dbe207442ea78d24dc471b4321a49e9c": "le code d'une grande bibliothèque se compte en centaines de milliers de lignes", "c22efcc7a4176c3eb5293e15ee05eb9c66b6909bca90744816e02982e5d53986": "le même appel fautif planterait aussi avec une autre bibliothèque", "e788be52358c95b1d502668b22a5988dae1d8b2a76755e3149b07729529a1e3c": "une erreur de forme ou de type revient à chaque exécution"}}
📝 Ex 23.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "e6c319a3b4ba0c84267b1fd736e30bdd40492deda73ed6c12952b0112d7daf6c", "id": "23.Q3b", "kind": "bool", "mistakes": {"d831c7e22f59973fbfd1267eb3bee435a3c01a02c13d3a91123a000a80d28e22": "relis la fin du §23.2.2 de la fiche"}}
📝 Ex 23.Q3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "0ce905647b64823797ed41490f1e9885093116b0db926cbdcd7ff91b08d45e3d", "id": "23.Q3c", "kind": "int", "magnitude"

**23.Q4 — Les trois sens du mot « modèle »**

In [6]:
wb.record("23.Q4a", "A", mistakes={"dessiné avant tout entraînement : y a-t-il des poids appris ?": "B",
          "on parle d'un réseau précis, pas d'une façon de programmer": "C"})
wb.record("23.Q4b", "B", mistakes={"l'envoi contient des poids appris": "A", "on parle d'un réseau précis, prêt à prédire": "C"})
wb.record("23.Q4c", "C", mistakes={"torch.nn, torch.optim et torch.utils.data ne sont pas un réseau": "A",
          "aucun poids appris n'est en jeu ici": "B"})
wb.record("23.Q4d", False, mistakes={"un state_dict contient des tenseurs rangés par nom : où est décrite l'architecture ?": True})
wb.record("23.Q4e", True, mistakes={"des formes compatibles disent seulement que le calcul s'exécute": False})

WB_ANSWER {"hash": "0283792212439e5634057c346758eb88c684a7fab7064758f1f8c0d6ba76dd3e", "id": "23.Q4a", "kind": "str", "mistakes": {"83b1a10933ac37764b331d6f2553de8beb029efe1185d946b6335f411b632be6": "dessiné avant tout entraînement : y a-t-il des poids appris ?", "fdf0509083739621cf4c69af65aeb6c8f35bb36b30d1645340c6f6a03c4ed286": "on parle d'un réseau précis, pas d'une façon de programmer"}}
📝 Ex 23.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "515242d2dc5f71f18ce5fd3a841863a2589ba3499567d2bfbe0ab7f9b99d910e", "id": "23.Q4b", "kind": "str", "mistakes": {"57435025a4086b884dedbcc51f75a13e7bafb37966df73b1899d69622ffcf9db": "l'envoi contient des poids appris", "af20d9f8d710223e268bca403c5439594d31689e2762412ae10ce6d3a8289c8d": "on parle d'un réseau précis, prêt à prédire"}}
📝 Ex 23.Q4b : réponse enregistrée (str).
WB_ANSWER {"hash": "9508341220dbf48f723988b729e07cb216575c343eb0364ba1b00abc05236ae2", "id": "23.Q4c", "kind": "str", "mistakes": {"5496a14f517b61ea633cc810f2db075c63347b9

**23.Q5 — Tenseurs, arrays et type float32**

In [7]:
wb.record("23.Q5a", 4, mistakes={"PyTorch garde l'axe des canaux (C = 1), en plus de N, H et W": 3,
          "un mini-batch d'images a plus de deux axes : N, C, H et W": 2})
wb.record("23.Q5b", "B", mistakes={"float16 ne sert qu'à la précision mixte, jamais par défaut": "A",
          "float64 est le défaut de NumPy, pas de PyTorch": "C",
          "uint8 est le type des pixels bruts, des entiers": "D"})
wb.record("23.Q5c", "C", mistakes={"from_numpy garde le type du tableau NumPy : quel est le défaut de NumPy ?": "B",
          "NumPy ne crée pas de float16 par défaut": "A", "0,5 n'est pas un entier": "D"})
wb.record("23.Q5d", True, mistakes={"from_numpy ne copie pas : modifie a et regarde le tenseur": False})
wb.record("23.Q5e", 4, mistakes={"32 bits, et un octet fait 8 bits": 32, "8 octets, c'est un float64": 8})

WB_ANSWER {"hash": "f096a897a0e1ad245787dd8ac637388c721a8ab9f2ae7c83cc948fef7f1167ff", "id": "23.Q5a", "kind": "int", "magnitude": 0, "mistakes": {"95fe4e036ad32e580d307127379c124582c0713a96afa504c089942a6535560e": "un mini-batch d'images a plus de deux axes : N, C, H et W", "d4e052d2f77b9b15efd61951fe3ef5b55833b58933cdb370f8635d84c96be654": "PyTorch garde l'axe des canaux (C = 1), en plus de N, H et W"}, "sign": 1}
📝 Ex 23.Q5a : réponse enregistrée (int).
WB_ANSWER {"hash": "3f9ff396d887aa1e92acb4a2fec20b16716b0cb20217a2d8c48a3782de5eaf10", "id": "23.Q5b", "kind": "str", "mistakes": {"b0d5a32420578fa01042e84826dce05dff0a2a53095c2242a3244bfa2e72ec2e": "uint8 est le type des pixels bruts, des entiers", "b55ece0d346680bcb5ae18cc2fccc0931e696e025dde7eae917903ce06bd7efa": "float16 ne sert qu'à la précision mixte, jamais par défaut", "fdd9ffc3cecbbcd0eea58cee5b1562e7bc0db98888eaf0bfb18c8d9ece874396": "float64 est le défaut de NumPy, pas de PyTorch"}}
📝 Ex 23.Q5b : réponse enregistrée (str).

**23.Q6 — channels_first, channels_last et NCHW**

In [8]:
wb.record("23.Q6a", "(3, 48, 64)", mistakes={"la hauteur (le nombre de lignes) vient avant la largeur": "(3, 64, 48)",
          "c'est channels_last : en channels_first, les canaux viennent d'abord": "(48, 64, 3)"})
wb.record("23.Q6b", "(48, 64, 3)", mistakes={"la hauteur (le nombre de lignes) vient avant la largeur": "(64, 48, 3)",
          "c'est channels_first : en channels_last, les canaux viennent à la fin": "(3, 48, 64)"})
wb.record("23.Q6c", "A", mistakes={"c'est le format channels_last de Keras": "B",
          "l'axe des exemples vient toujours en premier": "C",
          "la hauteur vient avant la largeur": "D"})
wb.record("23.Q6d", "(0, 3, 1, 2)", mistakes={"c'est la conversion inverse : donne, pour chaque axe du résultat, l'axe d'origine qui y va": "(0, 2, 3, 1)",
          "l'axe des exemples reste en position 0": "(3, 0, 1, 2)"})
wb.record("23.Q6e", False, mistakes={"pense à une image en couleur de 3 × 3 pixels": True})

WB_ANSWER {"hash": "1c7c96f24e4d46c315c03ab58092f084de3432c89c44e2ff30d8147c0f97531f", "id": "23.Q6a", "kind": "str", "mistakes": {"5762e6e3b8fc43a020a09ca7a385f343c681e16a1a90685fbe0fa209362f54f4": "la hauteur (le nombre de lignes) vient avant la largeur", "f64083be54c84a45d2d4dafa23b66a3f8b5c1374baa446765e8d329ce60e0b90": "c'est channels_last : en channels_first, les canaux viennent d'abord"}}
📝 Ex 23.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "28e0c9778270ad311f52e9e167fcc9c43c87256e6e49519619c0ba777ce3b872", "id": "23.Q6b", "kind": "str", "mistakes": {"438b2568db3d1785a688f42b3bc34ef4979114b2f86950b5fa98b8db2db891c7": "la hauteur (le nombre de lignes) vient avant la largeur", "a1c334a19577e41d5718f052f9096fb814b17d4ed23f261d528b090ff68f661d": "c'est channels_first : en channels_last, les canaux viennent à la fin"}}
📝 Ex 23.Q6b : réponse enregistrée (str).
WB_ANSWER {"hash": "d797f6849360152b9cbc00b34aedce3e4b1d37095ba9100602b2440d904eb3fc", "id": "23.Q6c", "kind": "str", "

**23.Q7 — CPU, GPU, MPS, TPU : où calcule PyTorch ?**

In [9]:
wb.record("23.Q7a", "C", mistakes={"cuda est le device des GPU NVIDIA (et AMD, avec ROCm)": "A",
          "Metal est le framework graphique d'Apple, mais le device porte un autre nom": "B",
          "un Mac n'a pas de TPU": "D"})
wb.record("23.Q7b", True, mistakes={"où se ferait le produit de matrices, si les deux tenseurs sont à deux endroits ?": False})
wb.record("23.Q7c", "C", mistakes={"l'horloge d'un GPU est plutôt plus lente que celle d'un CPU": "A",
          "un GPU a souvent moins de mémoire que l'ordinateur qui l'héberge": "B",
          "les GPU calculent en réels (float32, float16)": "D"})
wb.record("23.Q7d", True, mistakes={"y a-t-il un GPU dans un environnement d'exécution CPU ?": False})
wb.record("23.Q7e", False, mistakes={"relis l'encadré 🕰️ sur les accélérateurs : que faut-il pour un TPU ?": True})

WB_ANSWER {"hash": "f6780dd922d1a352837818a5dba1f28c6da008138bd9a78ac4711bd87b0c0201", "id": "23.Q7a", "kind": "str", "mistakes": {"9cf354bd7c9cfeb26400205caf514052851b37542911b219e141729a47592715": "un Mac n'a pas de TPU", "ba5806cf75fc6344da676c77e39376daa16b4bb5fabb7775df67abcd52beef3a": "cuda est le device des GPU NVIDIA (et AMD, avec ROCm)", "fcb7b25f8405b318c418a05e5923cce22e8e713d5dd63de9632e6f7594cc4787": "Metal est le framework graphique d'Apple, mais le device porte un autre nom"}}
📝 Ex 23.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "16c428704713fa4210e853ff0e2e77427a4a0e2d02dbb104fc314cfd08077241", "id": "23.Q7b", "kind": "bool", "mistakes": {"050a8776d99b7e68912e9c28ff7e8f2b7204257eed1f2346e87a2f0f85324c13": "où se ferait le produit de matrices, si les deux tenseurs sont à deux endroits ?"}}
📝 Ex 23.Q7b : réponse enregistrée (bool).
WB_ANSWER {"hash": "0780af0d9262e47bb7bad91b5121723b6d0824ff0dce0da1315916d208f92910", "id": "23.Q7c", "kind": "str", "mistakes": {"08f

**23.Q8 — Iris, MNIST, Fashion-MNIST : les « hello world » du ML**

In [10]:
wb.record("23.Q8a", 60000, mistakes={"70 000 est le total, train et test réunis": 70000, "10 000 est la taille du test": 10000})
wb.record("23.Q8b", 10)
wb.record("23.Q8c", "A", mistakes={"MNIST est toujours gratuit": "B",
          "c'est ImageNet qui a retiré des catégories pour des raisons éthiques, pas MNIST": "C",
          "MNIST est en niveaux de gris": "D"})
wb.record("23.Q8d", True, mistakes={"relis ce que veut dire « remplaçant direct »": False})
wb.record("23.Q8e", 4, mistakes={"deux parties de la fleur, deux mesures pour chacune": 2, "3 est le nombre d'espèces": 3})

WB_ANSWER {"hash": "a2e8312010f05107b1ab95d0ba25f9bebc806292aeee77ea670adbdeff44ee21", "id": "23.Q8a", "kind": "int", "magnitude": 4, "mistakes": {"4bb0a6357556bdfc123491956336223baf35b0befb74ce3d1878bf60f2bf2827": "70 000 est le total, train et test réunis", "c4972604e059c18e3dae69c928afcf915a205555c0acc9be0cea152fe3eaf1c3": "10 000 est la taille du test"}, "sign": 1}
📝 Ex 23.Q8a : réponse enregistrée (int).
WB_ANSWER {"hash": "a916ec97d1f66453d12e25f609eae784e8df3c6a832ee52290b57f4b4991a76c", "id": "23.Q8b", "kind": "int", "magnitude": 1, "sign": 1}
📝 Ex 23.Q8b : réponse enregistrée (int).
WB_ANSWER {"hash": "9daafa683ac06beeb82f2316cba4be9c425044f09c2310095632747694452e0f", "id": "23.Q8c", "kind": "str", "mistakes": {"b75b86c485abc343279c079664ef28496938b4487d6542424bf009ca14811798": "MNIST est toujours gratuit", "bba43e9e2fd7f713fe463a72ee574e87008c22e496d7d2b6b8bd29ecd222051b": "MNIST est en niveaux de gris", "e739ce448b3cd264eea8c39634873a10e63c675f7dc836a8fc29b267f0b0ef2c": "c'e

**23.Q9 — Labels entiers, one-hot et logits**

In [11]:
wb.record("23.Q9a", "B", mistakes={"la loss applique déjà sa propre softmax": "A",
          "les one-hot sont des cibles, pas des sorties du modèle": "C",
          "la loss calcule elle-même la log-softmax": "D"})
wb.record("23.Q9b", "B", mistakes={"les one-hot ne sont pas obligatoires : la loss prend des indices de classe": "A",
          "la loss ne lit pas de texte": "C", "des indices de classe sont des entiers (int64)": "D"})
wb.record("23.Q9c", False, mistakes={"une seconde softmax produit-elle une valeur interdite ?": True})
wb.record("23.Q9d", "(3, 10)", mistakes={"une ligne par label, une colonne par classe": "(10, 3)",
          "one_hot ajoute un axe de num_classes valeurs": "(3,)"})
wb.record("23.Q9e", "(3, 9)", mistakes={"sans num_classes, le nombre de classes est deviné d'après le plus grand label": "(3, 10)",
          "les classes commencent à 0 : de 0 à 8, il y en a 9": "(3, 8)"})

WB_ANSWER {"hash": "e562edfaf4574cc10353d50c5a5d95526dda46f6ac4e1927aea6c4cc5302abed", "id": "23.Q9a", "kind": "str", "mistakes": {"4c3a5e578a699a4305072cb92cd94f4f6708feb1a55a5bebd8a019365271348f": "la loss applique déjà sa propre softmax", "747a9026c8c0e5f51afcd8eebe393769e9dc4f66464f5eefe403bc3b2f3efacb": "la loss calcule elle-même la log-softmax", "9f4708575cb089851093d1d57abdda8740d5496cae506abef9a58aaed8c62f22": "les one-hot sont des cibles, pas des sorties du modèle"}}
📝 Ex 23.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "2e229695fe426c41f3da29bfb176f33dc3f2157a69208b33445fdcbdf35cdb37", "id": "23.Q9b", "kind": "str", "mistakes": {"840c42bc037d1585a2b742d8e95400ed997b46d16819ec9dbfda2cb7921b811a": "la loss ne lit pas de texte", "918b8723d05e083e4b9aa6a6ccf183a386796725bea7a2b8ebee7a2f19966841": "des indices de classe sont des entiers (int64)", "b5b97cfff5080b5c0055aedf56745718c823844f6cd2e127a805e3d08681b933": "les one-hot ne sont pas obligatoires : la loss prend des indi

**23.Q10 — fit() décortiqué : batch, epoch, validation, train() et eval()**

In [12]:
wb.record("23.Q10a", False, mistakes={"relis l'encadré « model.eval() ne coupe pas les gradients »": True})
wb.record("23.Q10b", "B", mistakes={"le learning rate est réglé par l'optimiseur": "A",
          "le device se change avec .to(device)": "C", "le nombre d'epochs est dans ta boucle": "D"})
wb.record("23.Q10c", "D", mistakes={"des images jamais vues pendant l'entraînement, mais qui servent à choisir": "A",
          "10 000 images donnent une mesure assez précise": "B",
          "Keras ne vérifie pas d'où viennent les données de validation": "C"})
wb.record("23.Q10d", True, mistakes={"relis le paragraphe sur loss.item() au §23.7": False})
wb.record("23.Q10e", "C", mistakes={"torch.compile accélère un modèle, il ne choisit ni loss ni optimiseur": "A",
          "eval() change le mode de certaines couches": "B", "to(device) déplace le modèle": "D"})

WB_ANSWER {"hash": "bf56a377c0e682f9fb6f5442e3bee50a01eee29b85dc8b3e20656ffae5de640e", "id": "23.Q10a", "kind": "bool", "mistakes": {"eca4d6e0d1c687fd365852b52aa3839e83b1fa44ccfaaa4e14c997f78fa4c771": "relis l'encadré « model.eval() ne coupe pas les gradients »"}}
📝 Ex 23.Q10a : réponse enregistrée (bool).
WB_ANSWER {"hash": "5aa290166ed454bef07080cfa554a3a5592d2714b91df1b5ebad0dd05c32cc97", "id": "23.Q10b", "kind": "str", "mistakes": {"7c28371d6913bd1d51af35c2bdf927878c121ab8ea43ee4ba4160ab1eb34157b": "le nombre d'epochs est dans ta boucle", "c648a64b4d8f20d3684b0edfa556dd56eff5ba0a0f7647535abe61f3845b971b": "le device se change avec .to(device)", "c9443803069d57de261a66eec681de6070e399e4dbd32427727504b4b9306bad": "le learning rate est réglé par l'optimiseur"}}
📝 Ex 23.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "ce21d69f7ef050f45c1a8bfeae4dad9bcd2ce8436e46ed239334f4373e2b5bb9", "id": "23.Q10c", "kind": "str", "mistakes": {"3e706fb282d8d3f4f2c2b46f6e9137940555ef1485474685a292

**23.Q11 — Sauvegarder tout, les poids ou l'architecture ?**

In [13]:
wb.record("23.Q11a", "C", mistakes={"sans l'état de l'optimiseur, Adam repartirait de moments nuls": "A",
          "l'objet modèle seul ne contient ni l'optimiseur, ni l'epoch, ni les générateurs": "B",
          "les prédictions ne permettent pas de continuer à apprendre": "D"})
wb.record("23.Q11b", "B", mistakes={"l'architecture n'est pas dans le state_dict : c'est le code": "A",
          "un state_dict ne contient aucun code": "C", "l'optimiseur a son propre state_dict": "D"})
wb.record("23.Q11c", False, mistakes={"weights_only=True existe justement pour empêcher cela": True})
wb.record("23.Q11d", 4, mistakes={"chaque couche Linear a deux tenseurs : ses poids et ses biais": 2,
          "Flatten et ReLU n'ont pas de paramètres": 8})
wb.record("23.Q11e", True, mistakes={"relis ce que vérifie load_state_dict": False})

WB_ANSWER {"hash": "378f3b09f563e1a3c660f2a1abdd58c3f37da87dbf1450d8d3121e7eb4214c0c", "id": "23.Q11a", "kind": "str", "mistakes": {"3ee9ac5f9fa58b3f2cbd59e00c5a79fce58c46167a44f28e541022387e8a654a": "les prédictions ne permettent pas de continuer à apprendre", "5d36e6670cdcfeffb034b5dea3909c81a9e7694ffce352b5f466f7251490d7b5": "l'objet modèle seul ne contient ni l'optimiseur, ni l'epoch, ni les générateurs", "f0bbcafd79d1b34b38045ef1a165bdcf89c211e7da1e9a00a788ca17ddf08004": "sans l'état de l'optimiseur, Adam repartirait de moments nuls"}}
📝 Ex 23.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "2f332b2a7689adf635fa617db49ebc8aab7f8474bdd75d06c5e889da26fa566a", "id": "23.Q11b", "kind": "str", "mistakes": {"70b07e9bb14efa0adde40d64ea97dd66cb6ccffeeccd904dbe481b542925229d": "un state_dict ne contient aucun code", "c86a2427fcd4a9a630ab96ce6cd7b44a6b4bf6eec6b07fceb49bd6345d5703c1": "l'architecture n'est pas dans le state_dict : c'est le code", "f3cacdf51e29e2779d24be9775083db39478941

**23.Q12 — 🕰️ Que devient le code Keras du livre en 2026 ?**

In [14]:
wb.record("23.Q12a", "B", mistakes={"ce sont les backends de Keras 2, en 2017": "A",
          "PyTorch n'est que l'un des backends de Keras 3": "C",
          "TensorFlow n'est que l'un des backends de Keras 3": "D"})
wb.record("23.Q12b", "A", mistakes={"predict_proba a été retiré lui aussi": "B", "to_categorical code des labels": "C",
          "fit entraîne le modèle": "D"})
wb.record("23.Q12c", False, mistakes={"compare ce que font les deux fonctions": True})
wb.record("23.Q12d", False, mistakes={"relis l'encadré 🕰️ du §23.3.3": True})
wb.record("23.Q12e", "C", mistakes={"to_yaml a été retiré pour des raisons de sécurité": "A",
          ".pt est l'extension habituelle de PyTorch": "B",
          ".h5 est un format ancien que Keras 3 lit encore": "D"})

WB_ANSWER {"hash": "c689800a58bfe396e286053760dfeda8a045f26d4611ec29ee3fe412524891f3", "id": "23.Q12a", "kind": "str", "mistakes": {"4de847e3337723a7e23c1294e579f836283089beb8be92b3b43510ad1c63b2de": "ce sont les backends de Keras 2, en 2017", "7b7080e2086af6a7357cb67ca13e6a04038c21019711b6ebb3f58c78e4e9ddec": "TensorFlow n'est que l'un des backends de Keras 3", "ee678e02cc3088e137e9608107219181693af7baaa60688bfb4a1a585f172dbd": "PyTorch n'est que l'un des backends de Keras 3"}}
📝 Ex 23.Q12a : réponse enregistrée (str).
WB_ANSWER {"hash": "4834ad61282436e35622cd92315881e4406e649dce215996df56a62cb9e21030", "id": "23.Q12b", "kind": "str", "mistakes": {"7c0f8ba1a32d2932757f4734204d3cc8e15db58c2e5d91b59c8e13e86f67fbf3": "fit entraîne le modèle", "87badc6bbbf2419531ac9e07e8037cde33f24143bf5cd9eccdcde4196f935cb0": "predict_proba a été retiré lui aussi", "bdb09ed0231620b14d3611de0ed7c6857f993537f6acd62a2e27b67fd2ff47e9": "to_categorical code des labels"}}
📝 Ex 23.Q12b : réponse enregistrée (s

**23.R1 — Ch. 22 : (N, T, D) et batch_first**

In [15]:
wb.record("23.R1a", "(32, 20, 50)", mistakes={"avec batch_first=True, les exemples viennent d'abord": "(20, 32, 50)",
          "le temps (20 tokens) vient avant les features (50 valeurs)": "(32, 50, 20)"})
wb.record("23.R1b", "(20, 32, 50)", mistakes={"sans batch_first, le temps vient d'abord": "(32, 20, 50)"})
wb.record("23.R1c", "(2, 32, 64)", mistakes={"h_n n'est jamais batch_first : (couches × sens, N, H)": "(32, 2, 64)",
          "deux couches : un état final par couche": "(1, 32, 64)",
          "H = 64, la taille de l'état, pas celle des entrées": "(2, 32, 50)"})
wb.record("23.R1d", False, mistakes={"relis la documentation de batch_first : concerne-t-il les états ?": True})
wb.record("23.R1e", True, mistakes={"compare la place de N dans les deux formes": False})

WB_ANSWER {"hash": "1c01e21359be14dd5e8a02db065e378a2b725eec3972c9843652b84ce4b40b68", "id": "23.R1a", "kind": "str", "mistakes": {"3314ce267d788a787c9df41c8f10f313887dfd379979fe4a5d03d3296b19522e": "le temps (20 tokens) vient avant les features (50 valeurs)", "ae9fe03db77b78049227da8ab92b581fafb1487da3aadde93c5f4123946fbca0": "avec batch_first=True, les exemples viennent d'abord"}}
📝 Ex 23.R1a : réponse enregistrée (str).
WB_ANSWER {"hash": "6f4c47d115e66ce2618cc69d36582f2b518db77fce95e7ad0abd5769bc2a4566", "id": "23.R1b", "kind": "str", "mistakes": {"71293b843fb9688578f72d2dba8af70f0c4da393b7bad1196ae476e13a2758a3": "sans batch_first, le temps vient d'abord"}}
📝 Ex 23.R1b : réponse enregistrée (str).
WB_ANSWER {"hash": "a93eedce81a2d99d46ecb067cd8b229e243ceea9a83b85ec272eac715769285b", "id": "23.R1c", "kind": "str", "mistakes": {"18d11e9da81b9bc8c6cf04278f955ca061bd688dd08bb4b759890d9227c7ae3b": "deux couches : un état final par couche", "5f395a8568bd5560dff3264301cee1cc05980ae5b5caa

**23.R2 — Ch. 20 : Les cinq lignes d'une étape d'entraînement**

In [16]:
wb.record("23.R2a", "BCEAD", mistakes={"il faut les gradients avant de mettre à jour les poids": "BCEDA",
          "il faut des logits avant de calculer la loss": "BECAD",
          "il faut une loss avant de calculer les gradients": "BCAED"})
wb.record("23.R2b", True, mistakes={"backward ajoute au .grad existant": False})
wb.record("23.R2c", "B", mistakes={"c'est le travail d'optimizer.step()": "A", "c'est le travail d'optimizer.zero_grad()": "C",
          "c'est le travail de loss_fn(logits, yb)": "D"})
wb.record("23.R2d", True, mistakes={"un tenseur de loss est relié à tout le graphe de calcul du mini-batch": False})
wb.record("23.R2e", "B", mistakes={"un planning par epoch avancerait alors une fois par mini-batch": "A",
          "PyTorch avertit si scheduler.step() précède optimizer.step()": "C",
          "l'optimiseur ne connaît pas le planning": "D"})

WB_ANSWER {"hash": "df592bcd7b3f2b1b77de894079615778e6feb566671fb1bbc8ada7ed5ec585d1", "id": "23.R2a", "kind": "str", "mistakes": {"c702c792caf5bdff8e02ca034a2d7467ed8755ab6580901e09128665032b8440": "il faut des logits avant de calculer la loss", "d3c85765f177bf2fd965de8a3875111af380debf7e6ae15536196b9ad8795938": "il faut les gradients avant de mettre à jour les poids", "e7664c691cafbae859fe6f07a93f1173516b75af69f6f731b0a73d0e948018a1": "il faut une loss avant de calculer les gradients"}}
📝 Ex 23.R2a : réponse enregistrée (str).
WB_ANSWER {"hash": "f8f99f035017e7a3affec3b33acc2f26245011c936f01481444e00cf8be1fd34", "id": "23.R2b", "kind": "bool", "mistakes": {"565a0ea22b189f0d1fb01a796e9d690919e2b6c55a403bb10fff90821037cb2e": "backward ajoute au .grad existant"}}
📝 Ex 23.R2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "f76a7daf82bec0622d70e79c5bea8f38cbbb4b6a87cdb2c572a77b419a0c9d94", "id": "23.R2c", "kind": "str", "mistakes": {"1b96972a6d152224a868230b32ece2ef78d7ec0ddb1a984d908dc

**23.R3 — Ch. 2 : Moyenne et écart-type du train pour normaliser les pixels**

In [17]:
wb.record("23.R3a", MU_R3, decimals=3, mistakes={"divise par 5, le nombre de pixels": 0.5})
wb.record("23.R3b", SIGMA_R3, decimals=3, mistakes={"avec le diviseur n (ddof=0), pas n − 1": SIGMA1_R3,
          "prends la racine de la variance": SIGMA_R3 ** 2})
wb.record("23.R3c", (0.9 - MU_R3) / SIGMA_R3, decimals=3, mistakes={"σ se calcule avec le diviseur n": (0.9 - MU_R3) / SIGMA1_R3,
          "soustrais d'abord la moyenne": 0.9 / SIGMA_R3})
wb.record("23.R3d", False, mistakes={"les statistiques de normalisation s'apprennent sur le train seul": True})
wb.record("23.R3e", "A", mistakes={"c'est une remise à l'échelle par le minimum et le maximum": "B",
          "c'est seulement la division par 255": "C", "c'est l'inverse de la normalisation": "D"})

WB_ANSWER {"decimals": 3, "hash": "433e2d691ffc1b93ec5137443a49f1a1c79ea7dd22fb70f709377b73f19e0c2c", "hash_coarse": "c69db681f1c3c0f6ca35517d0e79a313e067da3af399c3e516304c1f93915ecb", "id": "23.R3a", "kind": "float", "magnitude": -1, "mistakes": {"912a5e3d97a5c1e748bbcdcef89ecbdcbe8dde9b3e9d68e55d0eb2777e8f0c0f": "divise par 5, le nombre de pixels"}, "sign": 1}
📝 Ex 23.R3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "df31609b605834aff746f0e6a110b62c78a765bccc130838e0884b844e271b57", "hash_coarse": "54b5fe52efd5c96395aa39c29b530e384747cd6877d9626abc03a22f7eaa0326", "id": "23.R3b", "kind": "float", "magnitude": -1, "mistakes": {"81ba1b497cc150a7e2a30353f108d3dbb2dec17ae780cb37ac263754dec13175": "prends la racine de la variance", "c6f85ff1b69b69afecb6a58d9e554bffbe21fd417c1d549545d9598e2b588f26": "avec le diviseur n (ddof=0), pas n − 1"}, "sign": 1}
📝 Ex 23.R3b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "35ffb1c3933

**Ex 23.1 — Remodeler 12 éléments : indices et ordre de lecture**

In [18]:
wb.record("23.1a", str(LETTERS_231.reshape(3, 4)[2, 1]), mistakes={"le dernier indice varie le plus vite : position i × 4 + j": "R",
          "les indices commencent à 0": "Q", "la forme (3, 4) a 4 colonnes : une ligne avance de 4 positions": "S"})
wb.record("23.1b", str(LETTERS_231.reshape(4, 3)[2, 1]), mistakes={"la forme (4, 3) a 3 colonnes : position i × 3 + j": "V",
          "le dernier indice varie le plus vite": "S"})
wb.record("23.1c", str(LETTERS_231.reshape(2, 3, 2)[1, 2, 0]), mistakes={"ordre C : position i × 6 + j × 2 + k": "R"})
wb.record("23.1d", "(0, 1, 1)", mistakes={"ordre C : 4 = i × 6 + j × 3 + k": "(0, 0, 1)", "Q est à la position 4 (M est à 0)": "(0, 1, 2)"})
wb.record("23.1e", "AD", mistakes={"(4, 4) n'a pas 12 éléments non plus": "A", "(5, 3) n'a pas 12 éléments non plus": "D",
          "(12,) a bien 12 éléments": "ADE"})
wb.record("23.1f", np.arange(12).reshape(3, 4).T.reshape(12)[:4].tolist(), mistakes={"a.T n'est pas lu dans l'ordre de la mémoire : sa première ligne est la première colonne de a": [0, 1, 2, 3]})
wb.record("23.1g", flat_index((1, 2, 1), (2, 3, 2)), mistakes={"pour la forme (2, 3, 2), le deuxième indice avance de 2 positions, pas de 3": 13,
          "les positions commencent à 0": 12})

WB_ANSWER {"hash": "dc8729eb3a6b7ea213c50190ba63e86d04401377fbd3e199f69d309e77591f2e", "id": "23.1a", "kind": "str", "mistakes": {"21f9d784e8ecbc3e2df561792c315931a13acae3a49b24a7ca08b0fc36de7c22": "la forme (3, 4) a 4 colonnes : une ligne avance de 4 positions", "83917b4e92ab0f6c00ef0a48904305d802ff006c291ca3c299620eec84cd7561": "le dernier indice varie le plus vite : position i × 4 + j", "b2ea444e9fcbdfa13955ed6940ff472abb79b400daf290e943879f81e7074c60": "les indices commencent à 0"}}
📝 Ex 23.1a : réponse enregistrée (str).
WB_ANSWER {"hash": "a2970b09f5ec7208383e464b0ddedec601faa4108d2e0326d907c014b0763fbe", "id": "23.1b", "kind": "str", "mistakes": {"86ceccbdefaf85479bd17235e0e7a1107a199e6fea2e880176d2a8585bab473f": "la forme (4, 3) a 3 colonnes : position i × 3 + j", "e081248e969692adcf285992789fe03e9a39b3e739eac386f3611f0eaa17e06f": "le dernier indice varie le plus vite"}}
📝 Ex 23.1b : réponse enregistrée (str).
WB_ANSWER {"hash": "2fa6548770ff7a39ba5cb3e527d65752777d0e158ae75baa

**Ex 23.2 — De NHWC à NCHW : où atterrit chaque pixel ?**

In [19]:
wb.record("23.2a", flat_index((1, 2, 0), (2, 3, 3)), mistakes={"dans x, c'est le dernier indice (le canal) qui varie le plus vite : 9h + 3w + c": 5,
          "les positions commencent à 0": 16})
wb.record("23.2b", str(Y_232.shape), mistakes={"permute(2, 0, 1) met en tête l'axe 2 de x, puis son axe 0, puis son axe 1": "(3, 3, 2)"})
wb.record("23.2c", "(1, 2, 0)", mistakes={"y[c, h, w] = x[h, w, c] : le premier indice de y est le canal": "(0, 1, 2)",
          "ce sont les arguments de permute, pas des indices": "(2, 0, 1)"})
wb.record("23.2d", int(np.argwhere(Z_232.ravel() == Y_232[0, 1, 2])[0, 0]), mistakes={"contiguous a recopié les valeurs dans l'ordre de y : 6c + 3h + w": 15})
wb.record("23.2e", False, mistakes={"y est-il encore contigu après permute ?": True})
wb.record("23.2f", "B", mistakes={"reshape relit la mémoire de x dans l'ordre : w[0, 0, 1] est à la position 1": "A",
          "la position 1 de la mémoire est x[0, 0, 1] : quel canal ?": "C",
          "reshape(3, 2, 3) a bien 18 éléments, rien ne l'empêche": "D"})
wb.record("23.2g", "A", mistakes={"y[0, 0, 1] = x[0, 1, 0] : de quel pixel s'agit-il ?": "B",
          "y[0, 0, 1] = x[0, 1, 0] : quel pixel, quel canal ?": "C",
          "permute ne lève pas d'erreur": "D"})
wb.record("23.2h", 64 * 3 * 32 * 32 * 4, mistakes={"un float32 fait 4 octets": 64 * 3 * 32 * 32, "4 octets par valeur, pas 8": 64 * 3 * 32 * 32 * 8,
          "il y a 3 canaux": 64 * 32 * 32 * 4})

WB_ANSWER {"hash": "ca7abc395262a506d8648071c89f9bf73ac736cfb9ddc3cb439ff9fa83025205", "id": "23.2a", "kind": "int", "magnitude": 1, "mistakes": {"dccbd075117c3b46dce14053a0083d88cf187aa8d6213c4125c0ff1574078ad5": "les positions commencent à 0", "efa9f3ce0b3951d67adb2cd242d97bafcfe735cd767f964e7e95107185e137bf": "dans x, c'est le dernier indice (le canal) qui varie le plus vite : 9h + 3w + c"}, "sign": 1}
📝 Ex 23.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "1180c89f8703e0c954561c9b45f4a0edb9efbc6983dd25c14a1c7ddd3e341fa7", "id": "23.2b", "kind": "str", "mistakes": {"6eeec3d1f66e4787ea2a0ee0b91d125fd911af75821bf2658ac5f382ebd9052f": "permute(2, 0, 1) met en tête l'axe 2 de x, puis son axe 0, puis son axe 1"}}
📝 Ex 23.2b : réponse enregistrée (str).
WB_ANSWER {"hash": "97e0ca155f1e360ac7361a8835fac256b232af86b6f79bca5641e346c94ce8f4", "id": "23.2c", "kind": "str", "mistakes": {"816e3c3f8fddef9ac1d368e719ffe25b9d0c02c659ce6b011e8e757c7352c51d": "y[c, h, w] = x[h, w, c] : le premier

**Ex 23.3 — 623 290 paramètres : refaire le compte du livre**

In [20]:
wb.record("23.3a", dense(784, 784), mistakes={"n'oublie pas les 784 biais": 784 * 784})
wb.record("23.3b", dense(784, 10), mistakes={"n'oublie pas les 10 biais": 784 * 10})
wb.record("23.3c", dense(784, 784) + dense(784, 10), mistakes={"additionne les deux couches": dense(784, 784), "n'oublie pas les biais": 784 * 784 + 784 * 10})
wb.record("23.3d", dense(784, 784) / (dense(784, 784) + dense(784, 10)), decimals=3, mistakes={"a) divisé par c), pas l'inverse": (dense(784, 784) + dense(784, 10)) / dense(784, 784),
          "la part de la première couche, pas de la seconde": dense(784, 10) / (dense(784, 784) + dense(784, 10))})
wb.record("23.3e", dense(784, 256) + dense(256, 10), mistakes={"n'oublie pas les biais": 784 * 256 + 256 * 10,
          "la couche de sortie reçoit 256 valeurs, pas 784": dense(784, 256) + dense(784, 10),
          "n'oublie pas les 10 biais de la couche de sortie": dense(784, 256) + 256 * 10})
wb.record("23.3f", dense(784, 512) + dense(512, 256) + dense(256, 10), mistakes={"n'oublie pas les biais": 784 * 512 + 512 * 256 + 256 * 10,
          "n'oublie pas la couche de sortie": dense(784, 512) + dense(512, 256)})
wb.record("23.3g", 0, mistakes={"Flatten ne fait que changer la forme : il n'a rien à apprendre": 784})

WB_ANSWER {"hash": "b5cbb795bff84e5936a5d15fd71ea2ca468080534c5b3e756a8925172ee490fc", "id": "23.3a", "kind": "int", "magnitude": 5, "mistakes": {"1f1078aa88274450f307bd94ca3962cda3091a518c2ca82a0a5f99ee2e56422c": "n'oublie pas les 784 biais"}, "sign": 1}
📝 Ex 23.3a : réponse enregistrée (int).
WB_ANSWER {"hash": "f89f8f2d3688b00cf47e97f760c41a9d27b99eec6e3d59d2273367a5b4a9084b", "id": "23.3b", "kind": "int", "magnitude": 3, "mistakes": {"f2c7ff096a9d51d78187d31fe06fdb69f69227f917e8aaeebb8c7a5695036e46": "n'oublie pas les 10 biais"}, "sign": 1}
📝 Ex 23.3b : réponse enregistrée (int).
WB_ANSWER {"hash": "b538ae992f2c39af939762229de9670a7de8884fcccacbff103a585dc8cc3ffa", "id": "23.3c", "kind": "int", "magnitude": 5, "mistakes": {"5ac3272b9a6461c186e1a9c218f3285e08e6f5bfbe8cff7c7a620966b01ad43d": "n'oublie pas les biais", "f1ac049c5651e2ccb4f053976a15a263e57777e2043e264c90227d43542d83a0": "additionne les deux couches"}, "sign": 1}
📝 Ex 23.3c : réponse enregistrée (int).
WB_ANSWER {"decima

**Ex 23.4 — Une cross-entropy à la main : indice de classe ou one-hot**

In [21]:
wb.record("23.4a", P_234[1], decimals=3, mistakes={"la softmax prend des exponentielles, pas les logits eux-mêmes": 2 / 3,
          "la classe 1 est celle du logit 2 (les classes commencent à 0)": P_234[0]})
wb.record("23.4b", math.log(sum(math.exp(z) for z in Z_234)), decimals=3, mistakes={"le logarithme népérien, pas décimal": math.log10(sum(math.exp(z) for z in Z_234)),
          "le logarithme de la somme, pas la somme": sum(math.exp(z) for z in Z_234)})
wb.record("23.4c", cross_entropy(Z_234, 1), decimals=3, mistakes={"L = −z_y + ln Σ e^z : n'oublie pas le terme −z_y": cross_entropy(Z_234, 1) + 2,
          "la classe 1 est celle du logit 2": cross_entropy(Z_234, 0),
          "le logarithme népérien, pas décimal": -math.log10(P_234[1])})
wb.record("23.4d", True, mistakes={"avec la cible (0, 1, 0), un seul terme de la somme reste": False})
wb.record("23.4e", cross_entropy(P_234, 1), decimals=3, mistakes={"prends les probabilités de a) comme logits, puis recalcule −z₁ + ln Σ e^z": cross_entropy(Z_234, 1),
          "c'est la probabilité de la classe 1 après la seconde softmax ; la loss est son −ln": softmax(P_234)[1]})
wb.record("23.4f", (cross_entropy(Z_234, 1) + cross_entropy([0, 0, 0], 2)) / 2, decimals=3, mistakes={"la moyenne du mini-batch, pas la somme": cross_entropy(Z_234, 1) + cross_entropy([0, 0, 0], 2),
          "la seconde image a trois logits égaux : sa loss est ln 3": cross_entropy(Z_234, 1) / 2,
          "la loss du mini-batch est la moyenne des deux images, pas celle de la première": cross_entropy(Z_234, 1),
          "la loss du mini-batch est la moyenne des deux images, pas celle de la seconde": math.log(3)})
wb.record("23.4g", math.log(10), decimals=3, mistakes={"le logarithme népérien, pas décimal": 1.0, "−ln(1/10), pas 1/10": 0.1})

WB_ANSWER {"decimals": 3, "hash": "caeb33d88a6a8225480fe27feecff7f0a1d074c045e2b0c2f5163815ecb98954", "hash_coarse": "1745a9e5ceef741039b576a9b97e7ba2fda4afe5251089704093302def6039d7", "id": "23.4a", "kind": "float", "magnitude": -1, "mistakes": {"464589d274b8e1e7e0652034b43ef0bbbebf2361e7c56773ff4b458f1335568d": "la classe 1 est celle du logit 2 (les classes commencent à 0)", "cdd89a2ebf6f8a31feec53de86c7bc5df3a2bcee8a0937c76110dfde87707f7e": "la softmax prend des exponentielles, pas les logits eux-mêmes"}, "sign": 1}
📝 Ex 23.4a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "bde961a1c6988900ac60870c564a6bd2641468e637cf17b6f92978a59180df69", "hash_coarse": "99e2cb16c1cc052b30896f485e227a6cfd8154cf6ef4df5404930f8cf12fbb70", "id": "23.4b", "kind": "float", "magnitude": 0, "mistakes": {"c8b46b9df60d25553285c391559dcfba4ed86b68477b135c856ef6ae339fcde9": "le logarithme népérien, pas décimal", "eb04b94b20a644d03c1d55dcd497f4d9da6dffeca7e2dd655b1d914ad8bab2fe

**Ex 23.5 — Batches par epoch et nombre de mises à jour**

In [22]:
wb.record("23.5a", math.ceil(N_235 / 256), mistakes={"arrondis au-dessus : le dernier mini-batch, incomplet, compte aussi": N_235 // 256,
          "retire d'abord les 6 000 images de validation": math.ceil(60_000 / 256)})
wb.record("23.5b", N_235 // 256, mistakes={"avec drop_last, on arrondit en dessous": math.ceil(N_235 / 256)})
wb.record("23.5c", N_235 - (N_235 // 256) * 256, mistakes={"le dernier mini-batch contient le reste de la division, pas 256 images": 256})
wb.record("23.5d", math.ceil(N_235 / 256) * 20, mistakes={"sans drop_last, le dernier mini-batch incomplet compte aussi": (N_235 // 256) * 20,
          "n'oublie pas de retirer la validation": math.ceil(60_000 / 256) * 20})
wb.record("23.5e", math.ceil(60_000 / 256), mistakes={"arrondis au-dessus": 60_000 // 256})
wb.record("23.5f", math.ceil(12_000 / 64), mistakes={"arrondis au-dessus": 12_000 // 64})
wb.record("23.5g", (N_235 // 256) * 256, mistakes={"avec drop_last, le dernier mini-batch n'est pas vu": N_235})

WB_ANSWER {"hash": "0a9257f7d610db349197a454b44cc044d2e221156ff9ef2d84d07be1613dce2d", "id": "23.5a", "kind": "int", "magnitude": 2, "mistakes": {"6fbc3269662f3593acfe194952c8106f31ad8f9df379d9494ecd1be64a70654d": "retire d'abord les 6 000 images de validation", "d85e7789af32e302d5c7dcb035c0b10af537a25e72883010a333b9f4e2ba67b3": "arrondis au-dessus : le dernier mini-batch, incomplet, compte aussi"}, "sign": 1}
📝 Ex 23.5a : réponse enregistrée (int).
WB_ANSWER {"hash": "447c3bafae2422e681a9eb3c0f39f77cd1e0ac391aaa5a7fb71c6af8e89ddb8b", "id": "23.5b", "kind": "int", "magnitude": 2, "mistakes": {"945515c413b3794f354e57a0d4ed184bc318d5dd37d224d115d461da4835122d": "avec drop_last, on arrondit en dessous"}, "sign": 1}
📝 Ex 23.5b : réponse enregistrée (int).
WB_ANSWER {"hash": "de39b65c3abe39e55f41b32149621418f2fa4278e5665f23ed7525d4c3ba3b6b", "id": "23.5c", "kind": "int", "magnitude": 2, "mistakes": {"373f6ba9fd5cdf2a6e51048c7508ce3e9afad6f5305509a146e32e82c450353a": "le dernier mini-batch c

**Ex 23.6 — Early stopping à la main : patience et min_delta**

In [23]:
wb.record("23.6a", STOP_236A, mistakes={"0,379 < 0,38 : l'epoch 7 est une amélioration": 8,
          "on s'arrête quand le compteur atteint la patience, pas quand il la dépasse": 15})
wb.record("23.6b", BEST_236A, mistakes={"on recharge les poids de la dernière amélioration": 10,
          "0,379 < 0,38 : la dernière amélioration est plus tardive": 5})
wb.record("23.6c", STOP_236C, mistakes={"avec min_delta = 0,005, 0,379 n'est pas sous 0,38 − 0,005": 10})
wb.record("23.6d", BEST_236C, mistakes={"0,379 n'est pas une amélioration avec min_delta = 0,005": 7,
          "on recharge les poids de la dernière amélioration": 8})
wb.record("23.6e", STOP_236E, mistakes={"l'epoch 11 est une amélioration, qui remet le compteur à 0": 12})
wb.record("23.6f", BEST_236E, mistakes={"on recharge les poids de la dernière amélioration": 16,
          "l'epoch 11 (0,37) bat 0,379": 7})
wb.record("23.6g", STOP_236A, mistakes={"Keras 2.0 ne recharge rien : il garde les poids de la dernière epoch entraînée": 7})

WB_ANSWER {"hash": "695e992ff9eeb7951c5a9d47c881e0d165c2fbb2c74a7c87d4d371bbec09b5c9", "id": "23.6a", "kind": "int", "magnitude": 1, "mistakes": {"5c81116e5c2651b09b20b2e95633063877a3705e932c3468eee0ad88d475e3cb": "on s'arrête quand le compteur atteint la patience, pas quand il la dépasse", "66be5ff07473c29124c2dd0047ceda2744609cf2dacddef3027b401548f2bb26": "0,379 < 0,38 : l'epoch 7 est une amélioration"}, "sign": 1}
📝 Ex 23.6a : réponse enregistrée (int).
WB_ANSWER {"hash": "a19370b84826590d88b1d15b846bbce58ec4567240c09334e0b5fedca2555cc7", "id": "23.6b", "kind": "int", "magnitude": 0, "mistakes": {"0ed18f570ab1eb0b51baf938a8a52c413b0ebf829db1f12f61370fb3f9eba15b": "0,379 < 0,38 : la dernière amélioration est plus tardive", "1ed8d1f9831880a40e9241661498498d3aaf66e625b52c85ece2c43f03f53151": "on recharge les poids de la dernière amélioration"}, "sign": 1}
📝 Ex 23.6b : réponse enregistrée (int).
WB_ANSWER {"hash": "fdf6c8e067ce9bbf881b9938ba139f6147ae3602ac669219973d508ddc272462", "id":

**Ex 23.7 — Plannings de learning rate : le planning du livre avec LambdaLR**

In [24]:
wb.record("23.7a", 1.0 * book_lambda(25), decimals=2, mistakes={"1 − 0,01 × 25 = 0,75": 0.25,
          "pendant l'epoch 25, le planning a reçu 25 appels (epochs comptées depuis 0)": book_lambda(24)})
wb.record("23.7b", min(e for e in range(200) if book_lambda(e) <= 0.1), mistakes={"les epochs se comptent à partir de 0 dans cet exercice": 91})
wb.record("23.7c", 0.01 * book_lambda(50), decimals=4, mistakes={"LambdaLR multiplie λ par le learning rate de l'optimiseur": book_lambda(50),
          "pendant l'epoch 50, le planning a reçu 50 appels : λ(50)": 0.01 * book_lambda(49),
          "les epochs se comptent à partir de 0 : c'est λ(50) qu'il faut": 0.01 * book_lambda(51)})
wb.record("23.7d", round(book_lambda(0) / 0.05), mistakes={"il faut 0,05 × λ(0) = 1 : λ(0) = 1 / 0,05": 1})
wb.record("23.7e", 0.1 * 0.1 ** (65 // 30), decimals=4, mistakes={"⌊65/30⌋ = 2 paliers sont passés": 0.01, "deux paliers, pas trois": 0.0001})
wb.record("23.7f", 0.1 * (1 + math.cos(math.pi * 25 / 100)) / 2, decimals=4, mistakes={"le cosinus se calcule en radians": 0.1 * (1 + math.cos(math.radians(math.pi * 25 / 100))) / 2,
          "n'oublie pas de diviser 1 + cos par 2": 0.1 * (1 + math.cos(math.pi * 25 / 100))})
wb.record("23.7g", sum(book_lambda(e) <= 0.1 for e in range(100)), mistakes={"de 90 à 99 inclus": 9})

WB_ANSWER {"decimals": 2, "hash": "c30bd3428af0b2bd298af968370c7fa9618eb4fead03d44d09a2577e3d804343", "hash_coarse": "1c09770f160c033a4150587ca8ce12e9992a523fffb5c1bbfcc49a50409cbf4c", "id": "23.7a", "kind": "float", "magnitude": -1, "mistakes": {"a6f6862b5fff9d4da0c345e350d8381f5412870aee0ed7afe2e7875465dd96dc": "1 − 0,01 × 25 = 0,75", "bedfbb0fce946f99e137a9412ab278c63e33ff7d6f6b7b71292fadf9d1ef2901": "pendant l'epoch 25, le planning a reçu 25 appels (epochs comptées depuis 0)"}, "sign": 1}
📝 Ex 23.7a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "924e3ed29cad25ade980d011421775a9f44cbcb75f5b6ec96278da3d808c0044", "id": "23.7b", "kind": "int", "magnitude": 1, "mistakes": {"02d105b8fc342382e79f8cc3f594d03cdb1180b134df1b95c3a7b0c650542fca": "les epochs se comptent à partir de 0 dans cet exercice"}, "sign": 1}
📝 Ex 23.7b : réponse enregistrée (int).
WB_ANSWER {"decimals": 4, "hash": "655ed3253e5bbd2177b296c611625e079a5d37910879081fa9bb35f968887d20", "hash_coarse": "563

**Ex 23.8 — Fermi : durée d'une epoch et poids d'un checkpoint**

In [25]:
wb.record("23.8a", OPS_238 / 1e9, decimals=1, mistakes={"6 P N : la passe avant et la passe arrière": P_238 * N_238 / 1e9,
          "la passe arrière coûte environ le double de la passe avant : 6 P N, pas 2 P N": 2 * P_238 * N_238 / 1e9})
wb.record("23.8b", SECONDS_238, decimals=1, mistakes={"compte 6 P N opérations, pas P N": P_238 * N_238 / 50e9})
wb.record("23.8c", P_238 * 4 / 1e6, decimals=1, mistakes={"4 octets par float32": P_238 / 1e6, "un float32 fait 4 octets, pas 8": P_238 * 8 / 1e6,
          "en mégaoctets de 10⁶ octets, pas en Mio de 2²⁰": P_238 * 4 / 2 ** 20})
wb.record("23.8d", 3 * P_238 * 4 / 1e6, decimals=1, mistakes={"les poids et les deux moments d'Adam : trois tenseurs de la taille des poids": 2 * P_238 * 4 / 1e6,
          "le checkpoint garde les deux moments d'Adam, pas les gradients": 4 * P_238 * 4 / 1e6,
          "ajoute les deux moments d'Adam aux poids": P_238 * 4 / 1e6})
wb.record("23.8e", int(3600 // SECONDS_238), mistakes={"utilise la durée non arrondie de b)": 800})

WB_ANSWER {"decimals": 1, "hash": "516fc75aed0567db7c6756609bfc8b8485ce937141ffca801b5ba29873fddc9f", "hash_coarse": "286ea074f8466650f9f1751af3849f5bf4e300036b5290d6debdbe4fae8cce89", "id": "23.8a", "kind": "float", "magnitude": 2, "mistakes": {"8c96b0c80fd49dac1fc341b33aaa8045976c341314291c1b6c4277b104f4af79": "6 P N : la passe avant et la passe arrière", "93ca6786a7db439b088c2df92951ee369f06efece7af1ff68f6739ffe1e9290f": "la passe arrière coûte environ le double de la passe avant : 6 P N, pas 2 P N"}, "sign": 1}
📝 Ex 23.8a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "3260476160bd759acbeff489fdbab0b27be2b3bbc1933e846df2eb3bfa09282b", "hash_coarse": "5f77c8847ea5a7766909ef6031256d1d96c5b4cf5e6970f3b1f9d9dbac9bf681", "id": "23.8b", "kind": "float", "magnitude": 0, "mistakes": {"8a9a8b59e3b304107dfc537fada3678ff024328f67b1b96610b1ee46dfd96d8b": "compte 6 P N opérations, pas P N"}, "sign": 1}
📝 Ex 23.8b : réponse enregistrée (float, 1 décimale(s)).
WB_

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu préparer des images pour PyTorch (forme, type, normalisation, `DataLoader`) et expliquer pourquoi `reshape` ne remplace pas `permute` ?
2. Sais-tu écrire une boucle d'entraînement avec validation, et dire ce que font `model.eval()` et `torch.inference_mode()` ?
3. Sais-tu sauvegarder un modèle pour le reprendre ou le livrer, avec son prétraitement, et programmer un meilleur checkpoint, un planning et un early stopping ?

**Pour aller plus loin** : les tutoriels « Learn the Basics » de PyTorch, cités dans la fiche. La suite : le ch. 24, PyTorch en pratique 2 (CNN, RNN, augmentation de données).

*Les exercices de code 23.12 à 23.29 (parties A à D : le pipeline Fashion-MNIST, ta fonction `fit()`, la sauvegarde, le modèle pré-entraîné et les callbacks) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 23` pour obtenir le notebook complet.*